# EDA

## Libraries & Functions

In [0]:
#Call libraries
from pyspark.sql import functions as F

In [0]:
def get_cell_candidates(grid, row, col):
    """ Return valid candidates for an empty Sudoku cell. """

    if grid[row][col] != 0:
        return []

    #Collect numbers already present in the row
    row_values = set(grid[row])

    #Collect numbers already present in the column
    column_values = {
        grid[r][col]
        for r in range(9)
    }

    #Identify the starting position of the 3x3 box
    box_row = (row // 3) * 3
    box_col = (col // 3) * 3

    #Collect numbers already present in the 3x3 box
    box_values = {
        grid[r][c]
        for r in range(box_row, box_row + 3)
        for c in range(box_col, box_col + 3)
    }

    #Combine all restricted numbers
    used_numbers = row_values | column_values | box_values

    #Return valid candidates
    return sorted(set(range(1, 10)) - used_numbers)

## Puzzle structure

In [0]:
#Load validated Sudoku games
silver_df = spark.table("workspace.sudoku_silver.validated_games")

#Analyze the distribution of given cells
given_cells_distribution = (
    silver_df
    .groupBy("given_cells")
    .agg(F.count("*").alias("puzzle_count"))
    .orderBy("given_cells")
)

display(given_cells_distribution)

In [0]:
#Notes:
    #34 filled cells is the most frequent count, appearing in 455,004 puzzles (45.50%)
    #Puzzles with 33 to 35 filled cells account for approximately 91.79% of the dataset
    #The extremes, such as 29 or 37 filled cells, are very rare
    #The mean is approximately 33.81 filled cells

In [0]:
#Insight: 
    #Since the number of clues varies very little, it will be important to develop features that describe not just how many numbers there are, but also how they are distributed across the board


In [0]:
#Analyze puzzle structure
puzzle_structure_df = (
    silver_df
    .groupBy("given_cells", "empty_cells")
    .agg(F.count("*").alias("puzzle_count"))
    .withColumn("total_cells", F.col("given_cells") + F.col("empty_cells"))
    .orderBy("given_cells")
)

display(puzzle_structure_df)
#Attetion! Spark DataFrame is immutable

## Feature Engineering

### Row clue counts

In [0]:
#Convert puzzle strings into arrays
features_df = silver_df.withColumn(
    "puzzle_digits",
    F.split(F.col("quizzes"), "")
)

#Count filled cells in each row
features_df = features_df.withColumn(
    "row_clue_counts",
    F.expr("""
        transform(sequence(0, 8),
            i -> size(filter(slice(puzzle_digits, i * 9 + 1, 9),
                digit -> digit <> '0')))
    """)
)

display(
    features_df.select(
        "quizzes",
        "given_cells",
        "row_clue_counts"
    ).limit(5)
)

#The expression iterates across the nine rows of the Sudoku:
    #1. sequence(0, 8) -> Generates the row indices
    #2. slice() -> Extracts the nine numbers from each row
    #3. filter() -> Retains only the non-zero numbers
    #4. size() -> Counts the filled-in clues
    #5. transform() -> Returns an array containing nine counts

In [0]:
#Verify that row clue counts match the total given cells
invalid_row_counts = features_df.filter(
    F.expr("aggregate(row_clue_counts, 0, (acc, x) -> acc + x)")
    != F.col("given_cells")
).count()

print(f"Invalid row clue counts: {invalid_row_counts:,}")

In [0]:
#Calculate the average number of clues per row
features_df = features_df.withColumn(
    "row_clue_mean",
    F.col("given_cells") / F.lit(9.0)
)

#Calculate the population standard deviation of row clue counts
features_df = features_df.withColumn(
    "row_clue_std",
    F.expr("""
        sqrt(aggregate(row_clue_counts,CAST(0.0 AS DOUBLE),
            (acc, x) -> acc + pow(x - row_clue_mean, 2)) / 9.0
        )
    """)
)

display(
    features_df.select(
        "given_cells",
        "row_clue_counts",
        "row_clue_mean",
        "row_clue_std"
    ).limit(10)
)

#Notes:
    #Low standard deviation: the tracks are distributed relatively evenly
    #High standard deviation: some lines have many more tracks than others

In [0]:
#Summarize row clue dispersion across all puzzles
features_df.select(
    F.min("row_clue_std").alias("min_row_clue_std"),
    F.avg("row_clue_std").alias("avg_row_clue_std"),
    F.max("row_clue_std").alias("max_row_clue_std"),
    F.sum(
        F.when(F.col("row_clue_std").isNull(), 1).otherwise(0)
    ).alias("null_row_clue_std")
).show()

In [0]:
#Calculate standard deviation distribution percentiles
features_df.select(
    F.percentile_approx(
        "row_clue_std",
        [0.25, 0.50, 0.75, 0.95],
        10000
    ).alias("row_clue_std_percentiles")
).show(truncate=False)

In [0]:
#Notes:
    #Low dispersion: row_clue_std up to approximately 0.875
    #Intermediate dispersion: between 0.875 and 1.227 (~50%)
    #High dispersion: above 1.227
    #Very high dispersion: above 1.474, corresponding approximately to the top 5%

### Column clue counts

In [0]:
#Count filled cells in each column
features_df = features_df.withColumn(
    "column_clue_counts",
    F.expr("""
        transform(sequence(0, 8),
            i -> size(filter(transform(sequence(0, 8),
                j -> element_at(puzzle_digits, j * 9 + i + 1)),
                    digit -> digit <> '0')))
    """)
)

#Calculate the average and standard deviation
features_df = (
    features_df
    .withColumn("column_clue_mean", F.col("given_cells") / 9.0)
    .withColumn(
        "column_clue_std",
        F.expr("""
            sqrt(aggregate(column_clue_counts,CAST(0.0 AS DOUBLE),
                (acc, x) -> acc + pow(x - column_clue_mean, 2)) / 9.0)
        """)
    )
)

In [0]:
#Verify the total number of clues across columns
invalid_column_counts = features_df.filter(
    F.expr("""
        aggregate(column_clue_counts, 0, (acc, x) -> acc + x)
    """) != F.col("given_cells")
).count()

print(f"Invalid column clue counts: {invalid_column_counts:,}")

#Summarize column clue dispersion
features_df.select(
    F.min("column_clue_std").alias("min_column_clue_std"),
    F.avg("column_clue_std").alias("avg_column_clue_std"),
    F.max("column_clue_std").alias("max_column_clue_std")
).show()

In [0]:
#Insight: 
    #The distributions of clues across rows and columns exhibit virtually identical average dispersions


### Box clue counts

In [0]:
#Count filled cells in each 3x3 box
features_df = features_df.withColumn(
    "box_clue_counts",
    F.expr("""
        transform(sequence(0, 8),
            b -> size(filter(transform(sequence(0, 8),
                k -> element_at(puzzle_digits,CAST(((b DIV 3 * 3 + k DIV 3) * 9 + (b % 3 * 3 + k % 3) + 1 ) AS INT))),digit -> digit <> '0')))
    """)
)

In [0]:
#Calculate average clues and standard deviation across boxes
features_df = (
    features_df
    .withColumn("box_clue_mean", F.col("given_cells") / 9.0)
    .withColumn(
        "box_clue_std",
        F.expr("""
            sqrt(aggregate(box_clue_counts,CAST(0.0 AS DOUBLE),(acc, x) -> acc + pow(x - box_clue_mean, 2)) / 9.0)
        """)
    )
)

In [0]:
#Validate box clue counts
invalid_box_counts = features_df.filter(
    F.expr("""
        aggregate(box_clue_counts, 0, (acc, x) -> acc + x)
    """) != F.col("given_cells")
).count()

print(f"Invalid box clue counts: {invalid_box_counts:,}")

#Summarize box clue dispersion
features_df.select(
    F.min("box_clue_std").alias("min_box_clue_std"),
    F.avg("box_clue_std").alias("avg_box_clue_std"),
    F.max("box_clue_std").alias("max_box_clue_std")
).show()

In [0]:
#Insights
    #1. Similar distribution across rows and columns:
        #The averages are practically identical, indicating similar levels of clue dispersion in both directions

    #2. Blocks show slightly lower average dispersion:
        #The mean `box_clue_std` of 1.0059 indicates that clues tend to be distributed somewhat more evenly across 3×3 blocks than across rows or columns

    #3. Maximum values ​​reveal distinct patterns:
        #Although the average dispersion for blocks is lower, its maximum value is the highest among the three features
        #This suggests that there are boards where the distribution of clues across blocks is particularly uneven
        #These observations are descriptive. They do not yet demonstrate a relationship with the difficulty of the Sudoku puzzles

### Candidate Analysis

In [0]:
#Select one puzzle for candidate analysis
sample_puzzle = (
    features_df
    .select("quizzes", "given_cells")
    .limit(1)
    .collect()[0]
)

puzzle = sample_puzzle["quizzes"]

#Convert puzzle string into a 9x9 grid
grid = [
    [int(puzzle[row * 9 + col]) for col in range(9)]
    for row in range(9)
]

#Display the puzzle grid
for row in grid:
    print(row)

In [0]:
#First cell test
row = 0
col = 0

candidates = get_cell_candidates(grid, row, col)

print(f"Cell position: ({row}, {col})")
print(f"Valid candidates: {candidates}")
print(f"Candidate count: {len(candidates)}")

#Notes:
    #This is an example of a Naked Single: a cell that has only one possible candidate

In [0]:
#Collect candidates for all empty cells
cell_candidates = []

for row in range(9):
    for col in range(9):
        if grid[row][col] == 0:
            candidates = get_cell_candidates(grid, row, col)

            cell_candidates.append({
                "row": row,
                "col": col,
                "candidates": candidates,
                "candidate_count": len(candidates)
            })

#Display the first 10 empty cells
for cell in cell_candidates[:10]:
    print(cell)

In [0]:
#Calculate candidate-based features
candidate_counts = [
    cell["candidate_count"]
    for cell in cell_candidates
]

avg_candidate_count = (
    sum(candidate_counts) / len(candidate_counts)
    if candidate_counts else 0.0
)

single_candidate_cells = sum(
    count == 1 for count in candidate_counts
)

max_candidate_count = max(candidate_counts, default=0)

print(f"Average candidate count: {avg_candidate_count:.4f}")
print(f"Single candidate cells: {single_candidate_cells}")
print(f"Maximum candidate count: {max_candidate_count}")

In [0]:
#Validate candidate analysis
empty_cells_count = sum(
    cell == 0
    for row in grid
    for cell in row
)

print(f"Empty cells: {empty_cells_count}")
print(f"Analyzed cells: {len(cell_candidates)}")
print(f"Counts match: {empty_cells_count == len(cell_candidates)}")

#Check for empty cells without valid candidates
zero_candidate_cells = sum(
    count == 0 for count in candidate_counts
)

print(f"Zero-candidate cells: {zero_candidate_cells}")

In [0]:
#Insight: 
    #Out of the 46 empty cells, 14 have only one possible candidate. This accounts for approximately 30.4% of the empty cells
    #Therefore, this Sudoku puzzle offers several opportunities for immediate placement using the "Naked Single" technique
    #However, we cannot yet conclude that it is easy, as we do not know how many steps will be required to solve it completely

### Game creation

In [0]:
#Create an array containing the positions of empty cells
features_df = features_df.withColumn(
    "empty_positions",
    F.expr("""
        filter(sequence(0, 80),
            pos -> element_at(puzzle_digits, pos + 1) = '0'
        )
    """)
)

#Validate empty cell positions
invalid_empty_positions = features_df.filter(
    F.size("empty_positions") != F.col("empty_cells")
).count()

print(f"Invalid empty positions: {invalid_empty_positions:,}")

#Display sample results
display(
    features_df.select(
        "given_cells",
        "empty_cells",
        "empty_positions"
    ).limit(5)
)

In [0]:
#Calculate the number of valid candidates for each empty cell
features_df = features_df.withColumn(
    "candidate_counts",
    F.expr("""
        transform(
            empty_positions,
            pos -> size(
                array_except(
                    sequence(1, 9),
                    array_distinct(
                        concat(
                            transform(
                                sequence(0, 8),
                                i -> CAST(
                                    element_at(
                                        puzzle_digits,
                                        CAST((pos DIV 9) * 9 + i + 1 AS INT)
                                    ) AS INT
                                )
                            ),
                            transform(
                                sequence(0, 8),
                                i -> CAST(
                                    element_at(
                                        puzzle_digits,
                                        CAST(i * 9 + pos % 9 + 1 AS INT)
                                    ) AS INT
                                )
                            ),
                            transform(
                                sequence(0, 8),
                                i -> CAST(
                                    element_at(
                                        puzzle_digits,
                                        CAST(
                                            (
                                                (pos DIV 27 * 3 + i DIV 3) * 9
                                                + (pos % 9 DIV 3 * 3 + i % 3)
                                                + 1
                                            ) AS INT
                                        )
                                    ) AS INT
                                )
                            )
                        )
                    )
                )
            )
        )
    """)
)

#Notes:
    #transform() -> To iterate through the empty positions
    #element_at() -> To access the cells on the board
    #concat() -> To gather the values ​​found in the row, column, and block
    #array_distinct() -> To remove duplicate numbers
    #array_except() -> To determine which numbers from 1 to 9 are still available
    #size() -> To count the candidates

In [0]:
#Validate candidate count arrays
invalid_candidate_arrays = features_df.filter(
    F.size("candidate_counts") != F.col("empty_cells")
).count()

invalid_candidate_values = features_df.filter(
    F.expr("""
        exists(
            candidate_counts,
            count -> count < 1 OR count > 9
        )
    """)
).count()

print(f"Invalid candidate arrays: {invalid_candidate_arrays:,}")
print(f"Invalid candidate values: {invalid_candidate_values:,}")

In [0]:
#Calculate candidate-based features
features_df = (
    features_df
    .withColumn(
        "avg_candidate_count",
        F.expr("""
            aggregate(candidate_counts,CAST(0.0 AS DOUBLE),(acc, x) -> acc + x) / empty_cells
        """)
    )
    .withColumn(
        "single_candidate_cells",
        F.expr("""
            size(filter(candidate_counts,count -> count = 1))
        """)
    )
    .withColumn(
        "max_candidate_count",
        F.array_max("candidate_counts")
    )
)

In [0]:
#Retrieve the same sample puzzle used in Python
sample_features = (
    features_df
    .filter(F.col("quizzes") == puzzle)
    .select(
        "given_cells",
        "empty_cells",
        "avg_candidate_count",
        "single_candidate_cells",
        "max_candidate_count"
    )
)

display(sample_features)

In [0]:
#Compare Spark features with Python reference values
sample_row = sample_features.first()

validation_results = {
    "avg_candidate_count": abs(
        sample_row["avg_candidate_count"] - avg_candidate_count
    ) < 1e-9,
    "single_candidate_cells": (
        sample_row["single_candidate_cells"] == single_candidate_cells
    ),
    "max_candidate_count": (
        sample_row["max_candidate_count"] == max_candidate_count
    )
}

for feature, is_valid in validation_results.items():
    print(f"{feature}: {'PASS' if is_valid else 'FAIL'}")

In [0]:
#Summarize candidate-based features
candidate_summary_df = features_df.select(
    F.min("avg_candidate_count").alias("min_avg_candidates"),
    F.avg("avg_candidate_count").alias("mean_avg_candidates"),
    F.max("avg_candidate_count").alias("max_avg_candidates"),

    F.min("single_candidate_cells").alias("min_single_candidates"),
    F.avg("single_candidate_cells").alias("mean_single_candidates"),
    F.max("single_candidate_cells").alias("max_single_candidates"),

    F.min("max_candidate_count").alias("min_max_candidates"),
    F.avg("max_candidate_count").alias("mean_max_candidates"),
    F.max("max_candidate_count").alias("max_max_candidates")
)

display(candidate_summary_df)

In [0]:
#Insights:
    #On average, each empty cell has approximately 2.32 possible candidates
    #Each Sudoku puzzle contains, on average, 10.18 cells that can be filled immediately using the "Naked Single" technique
    #Some Sudoku puzzles have zero "Naked Single" candidates in their initial state. This means there are no "Naked Single" moves immediately available, though it does not necessarily imply that the puzzle is difficult
    #The maximum number of candidates found in a single cell ranges from 3 to 8, depending on the board

In [0]:
#Select final numerical features
feature_columns = [
    "given_cells",
    "row_clue_std",
    "column_clue_std",
    "box_clue_std",
    "avg_candidate_count",
    "single_candidate_cells",
    "max_candidate_count"
]

#Check for null values
null_summary_df = features_df.select([
    F.sum(
        F.when(F.col(column).isNull(), 1).otherwise(0)
    ).alias(column)
    for column in feature_columns
])

display(null_summary_df)

In [0]:
#Validate feature constraints
invalid_features = features_df.filter(
    (F.col("given_cells") < 0) |
    (F.col("given_cells") > 81) |
    (F.col("row_clue_std") < 0) |
    (F.col("column_clue_std") < 0) |
    (F.col("box_clue_std") < 0) |
    (F.col("avg_candidate_count") < 1) |
    (F.col("avg_candidate_count") > 9) |
    (F.col("single_candidate_cells") < 0) |
    (F.col("single_candidate_cells") > F.col("empty_cells")) |
    (F.col("max_candidate_count") < 1) |
    (F.col("max_candidate_count") > 9)
).count()

print(f"Invalid feature records: {invalid_features:,}")

### Create gold table

In [0]:
#Select the final Gold layer columns
gold_df = features_df.select(
    "quizzes",
    "given_cells",
    "row_clue_std",
    "column_clue_std",
    "box_clue_std",
    "avg_candidate_count",
    "single_candidate_cells",
    "max_candidate_count"
)

#Preview the Gold dataset
display(gold_df.limit(10))

In [0]:
#Save the Gold layer as a Delta table
gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.sudoku_gold.game_features")

print("Gold layer table created successfully.")

In [0]:
#Read the persisted Gold table
gold_validation_df = spark.table(
    "workspace.sudoku_gold.game_features"
)

#Validate record count and schema
print(f"Gold records: {gold_validation_df.count():,}")

gold_validation_df.printSchema()

#Check for duplicated puzzles
duplicate_puzzles = (
    gold_validation_df
    .groupBy("quizzes")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Duplicated puzzles: {duplicate_puzzles:,}")